In [ ]:
# 설치
!pip install transformers==4.45.0 -q
!pip install datasets -q
!pip install scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertModel

# ─── 1. Dual-Stream Prototype Manifold ────────────────────────────────────────
class PrototypeManifold(nn.Module):
    def __init__(self, embed_dim: int, n_prototypes: int = 32, temperature: float = 0.1):
        super().__init__()
        self.temperature = temperature
        half = n_prototypes // 2
        self.proto_s = nn.Parameter(torch.randn(half, embed_dim))
        self.proto_c = nn.Parameter(torch.randn(half, embed_dim))

    def _stream_stats(self, z, z_norm, proto):
        p_norm  = F.normalize(proto, dim=-1)
        sim     = z_norm @ p_norm.T
        max_sim, _ = sim.max(dim=-1)
        return max_sim

    def forward(self, z: torch.Tensor) -> dict:
        z_norm = F.normalize(z, dim=-1)

        support = self._stream_stats(z, z_norm, self.proto_s)
        counter = self._stream_stats(z, z_norm, self.proto_c)

        # ── prototype collapse 방지 ──────────────────────────────────────
        s_norm = F.normalize(self.proto_s, dim=-1)
        c_norm = F.normalize(self.proto_c, dim=-1)

        cross_div = (s_norm @ c_norm.T).abs().mean()

        K_s     = s_norm.size(0)
        K_c     = c_norm.size(0)
        gram_s  = s_norm @ s_norm.T
        gram_c  = c_norm @ c_norm.T
        eye_s   = torch.eye(K_s, device=z.device)
        eye_c   = torch.eye(K_c, device=z.device)
        intra_s = (gram_s - eye_s).pow(2).mean()
        intra_c = (gram_c - eye_c).pow(2).mean()

        diversity_loss = cross_div + 0.5 * (intra_s + intra_c)

        return {
            "support":         support,
            "counter":         counter,
            "diversity_loss":  diversity_loss,
        }

# ─── 2. Epistemic Field Classifier ────────────────────────────────────────────
class EpistemicFieldClassifier(nn.Module):

    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps        = eps
        self.scales     = nn.Parameter(torch.ones(6))
        self.truth_temp = nn.Parameter(torch.tensor(0.2))

        self.contra_bias = nn.Parameter(torch.tensor(1.0))  
        self.contra_temp = nn.Parameter(torch.tensor(0.2))

    def forward(self, manifold_out: dict, ignorance: torch.Tensor) -> dict:
        support         = manifold_out["support"]
        counter         = manifold_out["counter"]
        novelty_score   = manifold_out["novelty_score"]
        ambiguity_score = manifold_out["ambiguity_score"]

        temp     = F.softplus(self.truth_temp).clamp(min=0.05)
        margin_s = (support - counter).clamp(min=0.0)
        margin_c = (counter - support).clamp(min=0.0)

        # ── evidence plane ───────────────────────────────────────────────
        truth = support * torch.sigmoid(margin_s / temp)
        error = counter * torch.sigmoid(margin_c / temp)

        energy_sc     = support + counter
        agree         = (support - counter).abs().clamp(0.0, 1.0)
        c_temp        = F.softplus(self.contra_temp).clamp(min=0.05)
        contradiction = torch.sigmoid((energy_sc - F.softplus(self.contra_bias)) / c_temp) * (1.0 - agree)

        # ── independent uncertainty sources ──────────────────────────────
        novelty   = novelty_score
        ambiguity = ambiguity_score

        raw         = torch.stack(
            [truth, error, contradiction, novelty, ambiguity, ignorance], dim=-1
        )
        scales_norm = F.softmax(self.scales, dim=0) * 6.0
        field       = raw * scales_norm

        if self.training:
            dominant_type = None
            active_states = None
        else:
            dominant_idx  = field.argmax(dim=-1)
            dominant_type = [self.AXES[i] for i in dominant_idx.cpu().tolist()]
            active_states = []
            for sample in field:
                th     = sample.mean()
                active = [axis for idx, axis in enumerate(self.AXES) if sample[idx] > th]
                active_states.append(active if active else ["ignorance"])

        return {
            "field":         field,
            "dominant_type": dominant_type,
            "active_states": active_states,
            "truth":         truth,
            "error":         error,
            "contradiction": contradiction,
            "novelty":       novelty,
            "ambiguity":     ambiguity,
            "ignorance":     ignorance,
            "support_raw":   support,
            "counter_raw":   counter,
        }

# ─── Token-Novelty Source ─────────────────────────────────────────────────────
class TokenNovelty(nn.Module):
    """학습 어휘 대비 입력의 신규성 — 표면 형태, task/z와 완전 독립."""

    def __init__(self, vocab_size: int, eps: float = 1e-4):
        super().__init__()
        self.eps = eps
        self.register_buffer("token_count", torch.zeros(vocab_size))
        self.register_buffer("total", torch.tensor(0.0))
        # 특수토큰 마스킹용 (PAD/CLS/SEP은 신규성 계산서 제외)
        self.register_buffer("special_mask", torch.zeros(vocab_size, dtype=torch.bool))

    def set_special_tokens(self, ids):
        self.special_mask[torch.tensor(ids)] = True

    @torch.no_grad()
    def _update(self, input_ids, attention_mask):
        valid = input_ids[attention_mask.bool()]
        self.token_count.index_add_(0, valid, torch.ones_like(valid, dtype=torch.float))
        self.total += valid.numel()

    def forward(self, input_ids, attention_mask):
        if self.training:
            self._update(input_ids, attention_mask)

        # 토큰별 학습빈도 → 희귀도 = -log(freq), 미등장이면 최대
        total = self.total.clamp(min=1.0)
        freq  = self.token_count[input_ids] / total          # (B, T)
        rarity = -(freq + self.eps).log()                    # 희귀할수록 큼

        # 특수토큰·패딩 제외하고 문장 평균
        mask   = attention_mask.bool() & ~self.special_mask[input_ids]
        rarity = rarity * mask.float()
        denom  = mask.float().sum(dim=-1).clamp(min=1.0)
        sent_rarity = rarity.sum(dim=-1) / denom             # (B,)

        # EMA 없이 즉석 정규화 — log(total) 기준 (미등장 토큰 = -log(eps/total) 부근)
        max_rarity  = -torch.log(torch.tensor(self.eps, device=input_ids.device))
        novelty = (sent_rarity / max_rarity).clamp(0.0, 1.0)
        return novelty.detach()

# ─── Attention-based Ignorance Source ─────────────────────────────────────────
class AttentionIgnorance(nn.Module):

    def __init__(self, sep_id: int = 102, momentum: float = 0.01, eps: float = 1e-6):
        super().__init__()
        self.sep_id   = sep_id
        self.eps      = eps
        self.momentum = momentum
        self.register_buffer("mis_mean", torch.tensor(0.5))
        self.register_buffer("mis_std",  torch.tensor(0.15))
        self.register_buffer("initialized", torch.tensor(False))

    @torch.no_grad()
    def _update(self, d):
        m = self.momentum
        if not self.initialized:
            self.mis_mean.copy_(d.mean())
            self.mis_std.copy_(d.std() + self.eps)
            self.initialized.fill_(True)
        else:
            self.mis_mean.mul_(1 - m).add_(d.mean(), alpha=m)
            self.mis_std.mul_(1 - m).add_(d.std() + self.eps, alpha=m)

    def forward(self, attentions, input_ids, attention_mask):
        A = attentions[-1].detach().mean(dim=1)               # (B, S, S) head 평균
        is_sep     = (input_ids == self.sep_id)
        sep_cumsum = is_sep.cumsum(dim=1)
        amask      = attention_mask.bool()
        claim_mask = (sep_cumsum == 0) & amask                # CLS ~ 첫 SEP 전
        claim_mask[:, 0] = False                              # CLS 제외
        evid_mask  = (sep_cumsum == 1) & (~is_sep) & amask     # 첫 SEP 후 ~ 둘째 SEP 전

        evid_f  = evid_mask.unsqueeze(1).float()              # (B,1,S)
        mass    = (A * evid_f).sum(dim=-1)                    # (B,S) 각 query → evidence attention 질량
        claim_f = claim_mask.float()
        align   = (mass * claim_f).sum(dim=-1) / claim_f.sum(dim=-1).clamp(min=1.0)   # (B,) claim 토큰 평균 정렬
        mis     = 1.0 - align                                  # 정렬 부족 = ignorance 원신호

        if self.training:
            self._update(mis.detach())

        mu  = self.mis_mean.detach()
        std = self.mis_std.detach().clamp(min=self.eps)
        ignorance = torch.sigmoid((mis - mu) / std)
        return ignorance.detach()

# ─── Layer-Disagreement Ambiguity Source ──────────────────────────────────────
class LayerAmbiguity(nn.Module):
    """layer 간 [CLS] 방향 불일치 — 해석 비수렴. manifold(분류)와 독립 소스."""

    def __init__(self, momentum: float = 0.01, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.register_buffer("disp_mean", torch.tensor(0.3))
        self.register_buffer("disp_std",  torch.tensor(0.15))
        self.register_buffer("initialized", torch.tensor(False))

    @torch.no_grad()
    def _update(self, d):
        m = self.momentum
        if not self.initialized:
            self.disp_mean.copy_(d.mean())
            self.disp_std.copy_(d.std() + self.eps)
            self.initialized.fill_(True)
        else:
            self.disp_mean.mul_(1 - m).add_(d.mean(), alpha=m)
            self.disp_std.mul_(1 - m).add_(d.std() + self.eps, alpha=m)

    def forward(self, hidden_states):
        # hidden_states: tuple of (B, T, H), 임베딩 제외 후반 layer만 사용
        # 마지막 4개 layer의 [CLS] 방향 분산 (초기 layer는 표면적이라 제외)
        cls_layers = torch.stack(
            [h[:, 0] for h in hidden_states[-4:]], dim=1
        )                                                   # (B, L, H)
        cls_dir = F.normalize(cls_layers, dim=-1)            # 방향만
        mean_dir = F.normalize(cls_dir.mean(dim=1), dim=-1)  # (B, H) 평균 방향

        # 각 layer가 평균 방향에서 벗어난 각도 → 분산
        cos = (cls_dir * mean_dir.unsqueeze(1)).sum(dim=-1).clamp(-1, 1)  # (B, L)
        disp = (1.0 - cos).mean(dim=-1)                      # (B,) 평균 불일치 [0,2]

        if self.training:
            self._update(disp.detach())

        mu  = self.disp_mean.detach()
        std = self.disp_std.detach().clamp(min=self.eps)
        ambiguity = torch.sigmoid((disp - mu) / std)
        return ambiguity.detach()

        
# ─── 3. EpistemicBERT ─────────────────────────────────────────────────────────
class EpistemicBERT(nn.Module):

    def __init__(
        self,
        n_classes:    int  = 3,
        n_prototypes: int  = 32,
        proj_dim:     int  = 128,
        freeze_bert:  bool = False,
    ):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.bert.config.output_attentions = True
        self.bert.config.output_hidden_states = True
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(768, proj_dim)
        self.manifold  = PrototypeManifold(proj_dim, n_prototypes)
        self.token_nov = TokenNovelty(self.bert.config.vocab_size)
        self.attn_ign  = AttentionIgnorance() 
        self.layer_amb = LayerAmbiguity()
        self.epistemic = EpistemicFieldClassifier()

        self.field_proj = nn.Linear(6, n_classes)
        self.z_proj     = nn.Linear(proj_dim, n_classes)

        self.margin_param     = nn.Parameter(torch.tensor(0.2))
        self.energy_ceiling   = nn.Parameter(torch.tensor(0.5))
        self.con_energy_floor = nn.Parameter(torch.tensor(0.5))

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            output_attentions=True,                            # attention 활성화
        )
        cls = bert_out.last_hidden_state[:, 0]

        z             = self.proj(cls)
        manifold_out  = self.manifold(z)
        manifold_out["novelty_score"] = self.token_nov(input_ids, attention_mask)
        manifold_out["ambiguity_score"] = self.layer_amb(bert_out.hidden_states)

        # ── ignorance = attention dispersion (정보 결핍, 별도 소스) ────────
        ignorance = self.attn_ign(bert_out.attentions, input_ids, attention_mask)

        epistemic_out = self.epistemic(manifold_out, ignorance)

        field  = epistemic_out["field"]
        logits = self.field_proj(field) + self.z_proj(z)

        return logits, epistemic_out, manifold_out["diversity_loss"]

import torch
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import LinearRegression


@torch.no_grad()
def identifiability_probe(model, loader, device):
    model.eval()
    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    fields = []
    for batch in tqdm(loader, desc="train", mininterval=10.0, ncols=80):
        _, eout, _ = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        fields.append(eout["field"].cpu().float())
    F_mat = torch.cat(fields, dim=0).numpy()

    from sklearn.linear_model import LinearRegression
    import numpy as np

    # plane 축(0,1,2)과 uncertainty 축(3,4,5) 그룹 구분해서 해석
    print(f"\n{'axis':>16}  {'R² from others':>16}  {'group':>10}  {'verdict':>14}")
    print("─" * 62)
    groups = {0:"plane",1:"plane",2:"plane",3:"uncert",4:"uncert",5:"uncert"}
    for i, ax in enumerate(AXES):
        y   = F_mat[:, i]
        X   = np.delete(F_mat, i, axis=1)
        r2  = LinearRegression().fit(X, y).score(X, y)
        # plane 축은 redundant가 정상(2D 좌표계), uncert 축만 independent 기대
        if groups[i] == "plane":
            verdict = "plane-coord (OK)" if r2 > 0.6 else "unexpected-indep"
        else:
            verdict = "LEAK" if r2 > 0.6 else "independent (OK)"
        print(f"{ax:>16}  {r2:>16.4f}  {groups[i]:>10}  {verdict:>14}")

    corr = np.corrcoef(F_mat.T)
    print(f"\n  |correlation| matrix:")
    print(f"{'':>14}" + "".join(f"{a[:5]:>8}" for a in AXES))
    for i, ax in enumerate(AXES):
        row = "".join(f"{abs(corr[i, j]):>8.3f}" for j in range(6))
        print(f"{ax:>14}{row}")

import torch
import torch.nn.functional as F

# ─── Novelty / Ignorance Disentanglement Probe ────────────────────────────────
# 각 그룹은 (premise, hypothesis) — SNLI 입력 형식 유지
PROBE = {
    # G1: 정보 충분 + 문장 명확 + 구조가 낯섦 → novelty↑, ignorance↓
    "G1_novel_informed": [
        ("The quantum self-referential reasoner collapsed its own eigenstate.",
         "A self-modeling quantum device altered the particle it measured."),
        ("Zeta-7, discovered in 2029, decays into mirror-charged leptons.",
         "The Zeta-7 particle produces leptons with inverted charge."),
        ("The neuromorphic compiler hallucinated a non-Euclidean memory lattice.",
         "A brain-like compiler generated an impossible memory structure."),
        ("Synthetic ribozymes folded into a topology unseen in nature.",
         "Artificial RNA enzymes adopted a novel three-dimensional shape."),
        ("The exo-linguistic glyphs encoded a base-12 recursive grammar.",
         "The alien symbols followed a recursive twelve-base language system."),
    ],
    # G2: 정보 부족 + underspecified → ignorance↑, novelty↓
    "G2_ignorant_vague": [
        ("Something happened somewhere to someone.",
         "It probably turned out a certain way."),
        ("A particle was detected at some point.",
         "The thing was maybe important."),
        ("He said it might be the case, perhaps.",
         "The situation could possibly be relevant."),
        ("The truth about the event remains unknown.",
         "Nobody can say what really occurred."),
        ("They did the thing in the place that time.",
         "It was somehow related to the matter."),
    ],
    # G3: 평범한 익숙한 입력 → 둘 다 낮음 (기준선)
    "G3_ordinary": [
        ("A man is walking his dog in the park.",
         "A person is outside with an animal."),
        ("The woman bought groceries at the store.",
         "Someone purchased food at a shop."),
        ("Children are playing soccer on the field.",
         "Kids are playing a sport outdoors."),
        ("A chef is cooking pasta in the kitchen.",
         "A person is preparing food."),
        ("The train arrived at the station on time.",
         "A train reached its stop as scheduled."),
    ],
}


@torch.no_grad()
def novelty_ignorance_probe(model, tokenizer, device, max_length=128):
    model.eval()
    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    print(f"\n{'group':>20}  {'novelty':>9}  {'ignorance':>10}  {'nov-ign gap':>12}")
    print("─" * 58)

    group_means = {}
    for gname, pairs in PROBE.items():
        prem = [p for p, h in pairs]
        hyp  = [h for p, h in pairs]
        enc  = tokenizer(
            prem, hyp, max_length=max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        _, eout, _ = model(
            enc["input_ids"].to(device), enc["attention_mask"].to(device)
        )
        nov = eout["novelty"].mean().item()
        ign = eout["ignorance"].mean().item()
        group_means[gname] = {ax: eout[ax].mean().item() for ax in AXES}
        print(f"{gname:>20}  {nov:>9.4f}  {ign:>10.4f}  {nov - ign:>+12.4f}")

    # 핵심 판정: G1과 G2에서 두 축이 반대로 갈리는가
    g1, g2 = group_means["G1_novel_informed"], group_means["G2_ignorant_vague"]
    nov_sep = g1["novelty"]   - g2["novelty"]      # >0 이어야: G1이 더 novel
    ign_sep = g2["ignorance"] - g1["ignorance"]    # >0 이어야: G2가 더 ignorant

    print(f"\n  ── separation test ──")
    print(f"  novelty(G1) - novelty(G2)     = {nov_sep:>+.4f}   (>0 기대: G1이 더 낯섦)")
    print(f"  ignorance(G2) - ignorance(G1) = {ign_sep:>+.4f}   (>0 기대: G2가 정보부족)")

    if nov_sep > 0.03 and ign_sep > 0.03:
        print(f"  → 두 축이 의도대로 분리됨. nov↔ign 상관은 SNLI 데이터 아티팩트.")
    elif nov_sep > 0.03 or ign_sep > 0.03:
        print(f"  → 부분 분리. 한 축은 작동, 다른 축은 약함.")
    else:
        print(f"  → 분리 실패. 두 축이 같은 것을 측정 (측정 중복 가능성).")

    # 전체 field 프로파일 (각 그룹이 어느 축을 켜는지)
    print(f"\n  ── full field profile ──")
    print(f"{'group':>20}" + "".join(f"{a[:6]:>9}" for a in AXES))
    for gname, vals in group_means.items():
        print(f"{gname:>20}" + "".join(f"{vals[a]:>9.4f}" for a in AXES))

    return group_means

@torch.no_grad()
def attention_entropy_probe(model, tokenizer, device, max_length=128):
    model.eval()
    # DistilBERT attention 활성화
    model.bert.config.output_attentions = True

    print(f"\n{'group':>20}  {'attn_entropy':>13}  {'pred_entropy(ign)':>18}")
    print("─" * 56)

    for gname, pairs in PROBE.items():
        prem = [p for p, h in pairs]
        hyp  = [h for p, h in pairs]
        enc  = tokenizer(prem, hyp, max_length=max_length, padding="max_length",
                         truncation=True, return_tensors="pt")
        ids  = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)

        out  = model.bert(input_ids=ids, attention_mask=mask)
        # 마지막 layer, [CLS]가 각 토큰에 주는 attention, head 평균
        attn = out.attentions[-1]                       # (B, H, T, T)
        cls_attn = attn[:, :, 0, :].mean(dim=1)         # (B, T)  CLS→tokens, head mean
        # padding 마스킹 후 정규화
        cls_attn = cls_attn * mask
        cls_attn = cls_attn / (cls_attn.sum(dim=-1, keepdim=True) + 1e-8)
        ent      = -(cls_attn * (cls_attn + 1e-8).log()).sum(dim=-1)
        # 길이 정규화 (긴 문장이 자동으로 entropy 높으니)
        lengths  = mask.sum(dim=-1).float()
        ent_norm = ent / (lengths.log() + 1e-8)

        # 비교용 pred entropy (현재 ignorance)
        _, eout, _ = model(ids, mask)
        ign = eout["ignorance"].mean().item()

        print(f"{gname:>20}  {ent_norm.mean().item():>13.4f}  {ign:>18.4f}")

    model.bert.config.output_attentions = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast
from datasets import load_dataset
from tqdm.auto import tqdm
from collections import defaultdict
import json

# ─── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    model = dict(
        n_classes    = 3,
        n_prototypes = 32,
        proj_dim     = 128,
        freeze_bert  = False,
    ),
    train = dict(
        batch_size        = 32,        # 64 → 32 (max_length 늘려서 메모리 보전)
        epochs            = 3,      
        lr_bert           = 2e-5,
        lr_head           = 1e-3,
        max_length        = 192,       # 128 → 192 (evidence가 길어짐)
        lambda_ce         = 0.3,
        lambda_field      = 1.0,
        lambda_margin     = 0.2,
        lambda_diversity  = 0.1,
        ranking_margin    = 0.1,
        train_size        = 30_000,    # 50k → 30k (1 epoch 시간 단축)
        val_size          = 5_000,
        lambda_disent     = 0.03,
    ),
    device = "cuda" if torch.cuda.is_available() else "cpu",
    seed   = 42,
)


# ─── Dataset ──────────────────────────────────────────────────────────────────
class NLIDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data       = hf_split.filter(lambda x: x["label"] != -1)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["premise"], item["hypothesis"],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(item["label"], dtype=torch.long),
        }

class FEVERDataset(Dataset):
    LABEL_MAP = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}

    def __init__(self, hf_split, tokenizer, max_length):
        self.data       = hf_split
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        lbl  = item["label"]
        lbl  = self.LABEL_MAP[lbl] if isinstance(lbl, str) else int(lbl)
        enc  = self.tokenizer(
            item["claim"], item["evidence"],          # claim 먼저 → [CLS] claim [SEP] evidence [SEP]
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(lbl, dtype=torch.long),
        }

class OODDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data       = hf_split
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["sentence1"], item["sentence2"],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


# ─── Loss ─────────────────────────────────────────────────────────────────────
def field_ranking_loss(field, labels, margin: float = 0.1, eps: float = 1e-8):

    loss = torch.tensor(0.0, device=field.device)
    n    = torch.tensor(0,   device=field.device)

    sup_mask = (labels == 0)   # SUPPORTS → truth(0)
    ref_mask = (labels == 1)   # REFUTES  → error(1)
    nei_mask = (labels == 2)   # NEI      → ignorance(5)

    if sup_mask.any():
        f = field[sup_mask]
        for idx in [1, 2, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 0] - f[:, idx])).mean()
        n += sup_mask.sum()

    if ref_mask.any():
        f = field[ref_mask]
        for idx in [0, 2, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 1] - f[:, idx])).mean()
        n += ref_mask.sum()

    if nei_mask.any():
        f = field[nei_mask]
        for idx in [0, 1, 2, 3, 4]:
            loss += F.relu(margin - (f[:, 5] - f[:, idx])).mean()
        n += nei_mask.sum()

    return loss / (n.float() + eps)


def margin_loss(support, counter, labels, margin_param, energy_ceiling, con_energy_floor):
    margin  = F.softplus(margin_param).clamp(min=0.05)
    ceiling = F.softplus(energy_ceiling).clamp(min=0.2)

    loss     = torch.tensor(0.0, device=support.device)
    sup_mask = (labels == 0)   # SUPPORTS: support > counter
    ref_mask = (labels == 1)   # REFUTES : counter > support
    nei_mask = (labels == 2)   # NEI     : 둘 다 낮게 (관련 증거 없음)

    if sup_mask.any():
        loss += F.relu(counter[sup_mask] - support[sup_mask] + margin).mean()

    if ref_mask.any():
        loss += F.relu(support[ref_mask] - counter[ref_mask] + margin).mean()

    if nei_mask.any():
        energy = support[nei_mask] ** 2 + counter[nei_mask] ** 2
        loss  += F.relu(energy - ceiling).mean()

    return loss / 3.0


# ─── Optimizer ────────────────────────────────────────────────────────────────
def build_optimizer(model, cfg):
    bert_params = list(model.bert.parameters())
    head_params = (
        list(model.proj.parameters())
        + list(model.manifold.parameters())
        + list(model.epistemic.parameters())
        + list(model.field_proj.parameters())
        + list(model.z_proj.parameters())
        + [model.margin_param, model.energy_ceiling, model.con_energy_floor]
    )
    return torch.optim.AdamW([
        {"params": bert_params, "lr": cfg["lr_bert"]},
        {"params": head_params, "lr": cfg["lr_head"]},
    ], weight_decay=1e-2)


# ─── Train ────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scaler, device, tcfg):
    
    def disentangle_loss(eout):
        # 독립이어야 하는 모든 uncert 쌍 + uncert-plane 쌍
        def corr(a, b):
            a_c = a - a.mean()
            b_c = b - b.mean()
            return ((a_c * b_c).mean() / (a_c.std() * b_c.std() + 1e-8)).abs()
    
        nov = eout["novelty"]
        amb = eout["ambiguity"]
        ign = eout["ignorance"]
        con = eout["contradiction"]
    
        loss = (
            corr(nov, amb)
            + corr(amb, ign)
        )
        return loss / 2.0
    
    model.train()
    ce_fn = nn.CrossEntropyLoss()
    total_loss = total_ce = total_field = total_correct = total = 0

    for batch in tqdm(loader, desc="train", mininterval=10.0, ncols=80):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits, eout, div_loss = model(input_ids, attention_mask)

            ce     = ce_fn(logits, labels)
            f_loss = field_ranking_loss(eout["field"], labels, tcfg["ranking_margin"])
            m_loss = margin_loss(
                eout["support_raw"], eout["counter_raw"], labels,
                model.margin_param, model.energy_ceiling, model.con_energy_floor,
            )
            d_loss = disentangle_loss(eout)
            loss = (
                tcfg["lambda_ce"]        * ce
                + tcfg["lambda_field"]   * f_loss
                + tcfg["lambda_margin"]  * m_loss
                + tcfg["lambda_diversity"] * div_loss
                + tcfg["lambda_disent"]    * d_loss
            )

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  nan/inf  ce={ce.item():.4f}  f={f_loss.item():.4f}  m={m_loss.item():.4f}")
            break

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)
        total_loss    += loss.item()
        total_ce      += ce.item()
        total_field   += f_loss.item()

    n = len(loader)
    return {
        "loss":  total_loss  / n,
        "ce":    total_ce    / n,
        "field": total_field / n,
        "acc":   total_correct / total,
    }


# ─── Evaluate ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ce_fn = nn.CrossEntropyLoss()
    AXES  = EpistemicFieldClassifier.AXES

    axis_sums   = defaultdict(lambda: defaultdict(float))
    axis_counts = defaultdict(int)
    s_sums      = defaultdict(float)
    c_sums      = defaultdict(float)
    total_loss = total_correct = total = 0
    field_all  = []

    for batch in tqdm(loader, desc="train", mininterval=10.0, ncols=80):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)
            loss = ce_fn(logits, labels)

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item()
        total         += labels.size(0)
        field_all.append(eout["field"].cpu().float())

        for b in range(labels.size(0)):
            lbl = labels[b].item()
            axis_counts[lbl] += 1
            s_sums[lbl] += eout["support_raw"][b].item()
            c_sums[lbl] += eout["counter_raw"][b].item()
            for a, ax in enumerate(AXES):
                axis_sums[lbl][ax] += eout["field"][b, a].item()

    label_names = {0: "entailment", 1: "neutral", 2: "contradiction"}
    field_by_label = {
        label_names[lbl]: {ax: axis_sums[lbl][ax] / axis_counts[lbl] for ax in AXES}
        for lbl in label_names if axis_counts[lbl] > 0
    }
    field_cat = torch.cat(field_all, dim=0)
    monitor   = {
        "field_mean": field_cat.mean(0).numpy().round(4).tolist(),
        "field_std":  field_cat.std(0).numpy().round(4).tolist(),
    }
    return {
        "loss":           total_loss / len(loader),
        "acc":            total_correct / total,
        "field_by_label": field_by_label,
        "monitor":        monitor,
        "support_by_label": {
            label_names[lbl]: {
                "support": s_sums[lbl] / axis_counts[lbl],
                "counter": c_sums[lbl] / axis_counts[lbl],
            }
            for lbl in label_names if axis_counts[lbl] > 0
        },
    }


@torch.no_grad()
def evaluate_ood(model, ood_loader, id_loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES

    def collect(loader):
        fs = []
        for batch in tqdm(loader, desc="train", mininterval=10.0, ncols=80):
            with torch.cuda.amp.autocast():
                _, eout, _ = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                )
            fs.append(eout["field"].cpu())
        return torch.cat(fs, dim=0)

    id_f  = collect(id_loader)
    ood_f = collect(ood_loader)
    return {
        ax: {"id_mean": id_f[:, i].mean().item(), "ood_mean": ood_f[:, i].mean().item()}
        for i, ax in enumerate(AXES)
    }


@torch.no_grad()
def evaluate_confident_wrong(model, loader, device, top_frac=0.2):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES
    uncert_idx = [3, 4, 5]   # novelty, ambiguity, ignorance
    rows = []                # (correct, conf, entropy, field[6], label, pred)

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)

        prob  = F.softmax(logits.float(), dim=-1)
        preds = logits.argmax(dim=-1)

        conf = prob.max(dim=-1).values

        ent = -(prob * (prob + 1e-8).log()).sum(-1)
        ent = ent / torch.log(
            torch.tensor(float(prob.size(-1)), device=device)
        )

        field = eout["field"].float()

        for b in range(labels.size(0)):
            rows.append(
                (
                    (preds[b] == labels[b]).item(),
                    conf[b].item(),
                    ent[b].item(),
                    field[b].cpu().numpy(),
                    labels[b].item(),
                    preds[b].item(),
                )
            )

    import numpy as np

    correct = np.array([r[0] for r in rows], dtype=bool)
    conf    = np.array([r[1] for r in rows])
    ent     = np.array([r[2] for r in rows])
    fields  = np.stack([r[3] for r in rows])

    labels_arr = np.array([r[4] for r in rows])
    preds_arr  = np.array([r[5] for r in rows])

    def axis_means(mask):
        if mask.sum() == 0:
            return np.zeros(6)
        return fields[mask].mean(0)

    # --------------------------------------------------
    # correct / wrong / confident-wrong
    # --------------------------------------------------

    correct_m = axis_means(correct)
    wrong_m   = axis_means(~correct)

    wrong_idx = np.where(~correct)[0]

    if len(wrong_idx) > 0:
        wrong_conf = conf[wrong_idx]
        k = max(1, int(top_frac * len(wrong_idx)))

        conf_wrong_idx = wrong_idx[np.argsort(-wrong_conf)[:k]]

        cw_mask = np.zeros(len(correct), dtype=bool)
        cw_mask[conf_wrong_idx] = True
    else:
        cw_mask = np.zeros(len(correct), dtype=bool)

    confwrong_m = axis_means(cw_mask)

    print(f"\n{'axis':>14}  {'correct':>9}  {'all-wrong':>10}  {'confident-wrong':>16}")
    print("─" * 60)

    for i, ax in enumerate(AXES):
        print(
            f"{ax:>14}  "
            f"{correct_m[i]:>9.4f}  "
            f"{wrong_m[i]:>10.4f}  "
            f"{confwrong_m[i]:>16.4f}"
        )

    print(
        f"\n  confident-wrong: mean softmax conf = "
        f"{conf[cw_mask].mean():.4f}, "
        f"mean entropy = {ent[cw_mask].mean():.4f}"
    )

    print(
        f"  correct: mean softmax conf = "
        f"{conf[correct].mean():.4f}, "
        f"mean entropy = {ent[correct].mean():.4f}"
    )

    print(
        f"\n  error axis:"
        f" correct={correct_m[1]:.4f}"
        f" conf-wrong={confwrong_m[1]:.4f}"
        f" Δ={confwrong_m[1]-correct_m[1]:+.4f}"
    )

    print(
        f"  contradiction axis:"
        f" correct={correct_m[2]:.4f}"
        f" conf-wrong={confwrong_m[2]:.4f}"
        f" Δ={confwrong_m[2]-correct_m[2]:+.4f}"
    )

    # --------------------------------------------------
    # uncertainty silent wrong
    # --------------------------------------------------

    uncert_max = fields[~correct][:, uncert_idx].max(axis=1)

    correct_uncert_max = fields[correct][:, uncert_idx].max(axis=1)

    thresh = np.median(correct_uncert_max)

    silent = uncert_max < thresh

    print(
        f"\n  uncertainty threshold(correct median)"
        f" = {thresh:.4f}"
    )

    print(
        f"  silent wrong ratio = {silent.mean():.4f}"
        f" ({silent.sum()}/{len(silent)})"
    )

    # --------------------------------------------------
    # confident wrong label distribution
    # --------------------------------------------------

    label_names = {
        0: "entail",
        1: "neutral",
        2: "contra",
    }

    print("\n  confident-wrong label dist:", end="")

    for l in [0, 1, 2]:
        print(
            f" {label_names[l]}="
            f"{(labels_arr[cw_mask] == l).sum()}",
            end=""
        )

    print()

    # --------------------------------------------------
    # label controlled
    # --------------------------------------------------

    print("\n  label-controlled (contradiction / ambiguity):")

    for l in [0, 1, 2]:

        cw_l = cw_mask & (labels_arr == l)
        co_l = correct & (labels_arr == l)

        if cw_l.sum() > 0 and co_l.sum() > 0:

            print(
                f" {label_names[l]:>8}: "
                f"con cw={fields[cw_l][:,2].mean():.3f} "
                f"co={fields[co_l][:,2].mean():.3f}"
                f" | "
                f"amb cw={fields[cw_l][:,4].mean():.3f} "
                f"co={fields[co_l][:,4].mean():.3f}"
            )

    # --------------------------------------------------
    # HIGH CONFIDENCE ANALYSIS
    # --------------------------------------------------

    HIGH_CONF = 0.95

    hc_correct = correct & (conf > HIGH_CONF)
    hc_wrong   = (~correct) & (conf > HIGH_CONF)

    print("\n")
    print("=" * 60)
    print(f"HIGH CONFIDENCE ANALYSIS (conf > {HIGH_CONF})")
    print("=" * 60)

    print(
        f"high-conf correct = {hc_correct.sum()} | "
        f"high-conf wrong = {hc_wrong.sum()}"
    )

    hc_correct_m = axis_means(hc_correct)
    hc_wrong_m   = axis_means(hc_wrong)

    if hc_correct.sum() > 0 and hc_wrong.sum() > 0:

        print(
            f"\n{'axis':>14}  "
            f"{'HC-correct':>12}  "
            f"{'HC-wrong':>12}  "
            f"{'Δ(w-c)':>12}"
        )

        print("─" * 60)

        for i, ax in enumerate(AXES):

            delta = hc_wrong_m[i] - hc_correct_m[i]

            print(
                f"{ax:>14}  "
                f"{hc_correct_m[i]:>12.4f}  "
                f"{hc_wrong_m[i]:>12.4f}  "
                f"{delta:>+12.4f}"
            )

        print(
            f"\nHC-correct mean conf = "
            f"{conf[hc_correct].mean():.4f}"
        )

        print(
            f"HC-wrong mean conf = "
            f"{conf[hc_wrong].mean():.4f}"
        )

        print(
            f"\nAMBIGUITY DELTA = "
            f"{hc_wrong_m[4] - hc_correct_m[4]:+.4f}"
        )

        print(
            f"ERROR DELTA = "
            f"{hc_wrong_m[1] - hc_correct_m[1]:+.4f}"
        )

        print(
            f"IGNORANCE DELTA = "
            f"{hc_wrong_m[5] - hc_correct_m[5]:+.4f}"
        )

    else:
        print(
            "\nNot enough high-confidence wrong samples."
        )

@torch.no_grad()
def failure_typing(model, loader, tokenizer, device, conf_thresh=0.95, k_examples=3):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES
    unc_idx = {"novelty": 3, "ambiguity": 4, "ignorance": 5}

    rows = []   # (conf, ent, field[6], correct, ids, label, pred)
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbl  = batch["label"].to(device)
        with torch.cuda.amp.autocast():
            logits, eout, _ = model(ids, mask)
        prob  = F.softmax(logits.float(), dim=-1)
        preds = logits.argmax(dim=-1)
        conf  = prob.max(dim=-1).values
        ent   = -(prob * (prob + 1e-8).log()).sum(-1) / torch.log(
            torch.tensor(float(prob.size(-1)), device=device))
        fld   = eout["field"].float()
        for b in range(lbl.size(0)):
            rows.append((
                conf[b].item(), ent[b].item(), fld[b].cpu().numpy(),
                (preds[b] == lbl[b]).item(),
                ids[b].cpu(), lbl[b].item(), preds[b].item(),
            ))

    import numpy as np
    conf_a = np.array([r[0] for r in rows])
    ent_a  = np.array([r[1] for r in rows])
    fld_a  = np.stack([r[2] for r in rows])
    corr_a = np.array([r[3] for r in rows], dtype=bool)

    # confident wrong 집합
    cw = (~corr_a) & (conf_a > conf_thresh)
    n_cw = int(cw.sum())
    print(f"\nconfident wrong (conf>{conf_thresh}): {n_cw}개")
    if n_cw < 6:
        print("  표본 너무 적음 — conf_thresh 낮추거나 epoch 더."); return

    cw_fld  = fld_a[cw]
    cw_conf = conf_a[cw]
    cw_ent  = ent_a[cw]

    # 각 샘플을 "가장 높은 uncertainty 축"으로 유형 배정 (셋 다 낮으면 metacognitive)
    LOW = np.median(fld_a[corr_a][:, list(unc_idx.values())], axis=0)  # 맞은것의 축별 중앙값 = 기준선
    types = []
    for s in cw_fld:
        u = {ax: s[i] for ax, i in unc_idx.items()}
        top_ax = max(u, key=u.get)
        # top 축이 기준선(맞은것 중앙값) 넘어야 그 유형, 셋 다 못 넘으면 metacognitive
        if u[top_ax] > LOW[list(unc_idx).index(top_ax)]:
            types.append(top_ax)
        else:
            types.append("metacognitive")
    types = np.array(types)

    # ── 유형별 분포 + 각 유형이 conf로는 안 갈린다는 것 ──
    print(f"\n{'유형':>14}  {'n':>4}  {'conf':>7}  {'entropy':>8}  "
          f"{'novel':>7}  {'ambig':>7}  {'ignor':>7}  {'error':>7}")
    print("─" * 76)
    for t in ["ignorance", "ambiguity", "novelty", "metacognitive"]:
        m = types == t
        if m.sum() == 0: continue
        f = cw_fld[m]
        print(f"{t:>14}  {m.sum():>4}  {cw_conf[m].mean():>7.3f}  {cw_ent[m].mean():>8.3f}  "
              f"{f[:,3].mean():>7.3f}  {f[:,4].mean():>7.3f}  {f[:,5].mean():>7.3f}  {f[:,1].mean():>7.3f}")

    # ── 성공기준 2: conf/entropy가 유형을 못 가른다 ──
    print(f"\n  ── conf/entropy로 유형 분리되나 (안 돼야 우리 주장 성립) ──")
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        m = types == t
        if m.sum() > 0:
            print(f"    {t:>14}: conf={cw_conf[m].mean():.3f}±{cw_conf[m].std():.3f}  "
                  f"ent={cw_ent[m].mean():.3f}±{cw_ent[m].std():.3f}")
    print(f"    → 유형 간 conf 거의 같은데 6축은 다르면: 단일 confidence가 못 보는 걸 분해가 본다")

    # ── 성공기준 3: 유형별 실제 예시 텍스트 ──
    LABELS = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    print(f"\n  ── 유형별 실제 사례 (claim ‖ evidence) ──")
    cw_rows = [r for r, m in zip(rows, cw) if m]
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        idxs = np.where(types == t)[0]
        if len(idxs) == 0: continue
        # 그 유형 축이 가장 강한 순으로 정렬해 대표 사례
        if t == "metacognitive":
            order = idxs[np.argsort(cw_conf[idxs])[::-1]]      # 가장 자신만만한 순
        else:
            ax_i = unc_idx[t]
            order = idxs[np.argsort(cw_fld[idxs][:, ax_i])[::-1]]
        print(f"\n  [{t}]")
        for j in order[:k_examples]:
            r = cw_rows[j]
            text = tokenizer.decode(r[4], skip_special_tokens=True)
            text = text[:160] + ("…" if len(text) > 160 else "")
            print(f"    gold={LABELS[r[5]]:>8} pred={LABELS[r[6]]:>8} conf={r[0]:.3f}  "
                  f"nov={r[2][3]:.2f} amb={r[2][4]:.2f} ign={r[2][5]:.2f}")
            print(f"      {text}")

    return {"types": types, "cw_fld": cw_fld, "cw_conf": cw_conf}

@torch.no_grad()
def evaluate_selective_prediction(model, loader, device):
    model.eval()

    correct_all = []
    acc = {                       # 각 score 후보를 배치별로 누적
        "entropy (baseline)": [],
        "1-MSP (baseline)":   [],
        "ambiguity (ours)":   [],
        "error (ours)":       [],
        "amb+error (ours)":   [],
        "amb+ign+nov (ours)": [],
        "all-unc+error (ours)": [],
    }

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)

        prob  = F.softmax(logits.float(), dim=-1)
        preds = logits.argmax(dim=-1)
        n_cls = prob.size(-1)

        amb = eout["ambiguity"].float()
        ign = eout["ignorance"].float()
        nov = eout["novelty"].float()
        err = eout["error"].float()

        # baselines (높을수록 불확실)
        ent = -(prob * (prob + 1e-8).log()).sum(dim=-1)
        ent = ent / torch.log(torch.tensor(float(n_cls), device=device))
        msp = 1.0 - prob.max(dim=-1).values

        correct_all.append((preds == labels).cpu())
        acc["entropy (baseline)"].append(ent.cpu())
        acc["1-MSP (baseline)"].append(msp.cpu())
        acc["ambiguity (ours)"].append(amb.cpu())
        acc["error (ours)"].append(err.cpu())
        acc["amb+error (ours)"].append((amb + err).cpu())
        acc["amb+ign+nov (ours)"].append((amb + ign + nov).cpu())
        acc["all-unc+error (ours)"].append((amb + ign + nov + err).cpu())

    correct = torch.cat(correct_all).numpy().astype(float)
    scores  = {name: torch.cat(parts).numpy() for name, parts in acc.items()}
    return _risk_coverage_report(correct, scores)

@torch.no_grad()
def evaluate_fever_axes(model, loader, device):
    model.eval()
    AXES   = EpistemicFieldClassifier.AXES
    LABELS = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    fields, labels = [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast():
            _, eout, _ = model(ids, mask)
        fields.append(eout["field"].float().cpu())
        labels.append(batch["label"])
    import numpy as np
    fld = torch.cat(fields).numpy()
    lab = torch.cat(labels).numpy()
    means = {l: fld[lab == l].mean(0) for l in [0, 1, 2] if (lab == l).any()}

    print(f"\n{'axis':>14}" + "".join(f"{LABELS[l]:>11}" for l in [0, 1, 2]))
    print("─" * 50)
    for i, ax in enumerate(AXES):
        row = "".join(f"{means[l][i]:>11.4f}" if l in means else f"{'-':>11}" for l in [0, 1, 2])
        print(f"{ax:>14}{row}")

    def hi(axis_i, target):
        vals = {l: means[l][axis_i] for l in means}
        top  = max(vals, key=vals.get)
        return f"{'✓' if top == target else '✗'} (top={LABELS[top]} {vals[top]:.3f})"
    print(f"\n  매핑 점검:")
    print(f"    truth     → SUPPORTS  {hi(0, 0)}")
    print(f"    error     → REFUTES   {hi(1, 1)}")
    print(f"    ignorance → NEI       {hi(5, 2)}")

    print(f"\n  novelty vs ignorance (라벨별):")
    for l in [0, 1, 2]:
        if l in means:
            print(f"    {LABELS[l]:>9}: novelty={means[l][3]:.4f}  ignorance={means[l][5]:.4f}")

    print(f"\n  |corr| ignorance vs 나머지 (분리 확인):")
    for i, ax in enumerate(AXES):
        if ax == "ignorance": continue
        c = abs(np.corrcoef(fld[:, 5], fld[:, i])[0, 1])
        print(f"    ignorance ↔ {ax:>14}: {c:.4f}")
    return means

@torch.no_grad()
def diagnose_ambiguity(model, loader, device):
    """ambiguity가 plane으로 새는 게 정규화 포화 때문인지 진단.
    raw disp 분포 vs 정규화 후(field) 분포 + EMA 버퍼 상태를 본다."""
    model.eval()
    la = model.layer_amb

    # 현재 EMA 버퍼 상태
    print(f"\n  LayerAmbiguity EMA 버퍼:")
    print(f"    disp_mean = {la.disp_mean.item():.4f}")
    print(f"    disp_std  = {la.disp_std.item():.4f}   (< 0.03이면 포화 의심)")
    print(f"    initialized = {la.initialized.item()}")

    # raw disp를 직접 다시 계산 (forward 로직 복제, 정규화 전)
    import numpy as np
    raw_disps, amb_out, errs, contras = [], [], [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast():
            out = model.bert(input_ids=ids, attention_mask=mask)
            _, eout, _ = model(ids, mask)
        hs = out.hidden_states
        cls_layers = torch.stack([h[:, 0] for h in hs[-4:]], dim=1)
        cls_dir  = F.normalize(cls_layers.float(), dim=-1)
        mean_dir = F.normalize(cls_dir.mean(dim=1), dim=-1)
        cos  = (cls_dir * mean_dir.unsqueeze(1)).sum(dim=-1).clamp(-1, 1)
        disp = (1.0 - cos).mean(dim=-1)
        raw_disps.append(disp.cpu().numpy())
        amb_out.append(eout["ambiguity"].float().cpu().numpy())
        errs.append(eout["error"].float().cpu().numpy())
        contras.append(eout["contradiction"].float().cpu().numpy())

    raw = np.concatenate(raw_disps)
    amb = np.concatenate(amb_out)
    err = np.concatenate(errs)
    con = np.concatenate(contras)

    print(f"\n  raw disp (정규화 전, task-free 신호):")
    print(f"    mean={raw.mean():.4f}  std={raw.std():.4f}  "
          f"min={raw.min():.4f}  max={raw.max():.4f}")
    print(f"  ambiguity (정규화 후, field):")
    print(f"    mean={amb.mean():.4f}  std={amb.std():.4f}   (< 0.05면 거의 상수=포화)")

    # 핵심: raw disp가 error/contra와 상관있나? (소스 본질 결함)
    #       vs 정규화 후만 상관있나? (정규화가 만든 인공 상관)
    def corr(a, b):
        return abs(np.corrcoef(a, b)[0, 1])
    print(f"\n  ── raw disp의 plane 상관 (소스 본질 결함 여부) ──")
    print(f"    raw_disp ↔ error  = {corr(raw, err):.4f}")
    print(f"    raw_disp ↔ contra = {corr(raw, con):.4f}")
    print(f"  ── 정규화 후 ambiguity의 plane 상관 ──")
    print(f"    ambiguity ↔ error  = {corr(amb, err):.4f}")
    print(f"    ambiguity ↔ contra = {corr(amb, con):.4f}")
    print(f"\n  판정:")
    print(f"    raw도 상관 높음(>0.5) → 소스 본질 결함 (layer disagreement가 FEVER서 plane과 엮임)")
    print(f"    raw는 낮은데 정규화 후만 높음 → 정규화 포화가 인공 상관 생성 (EMA 손보면 됨)")


def _risk_coverage_report(correct, scores):
    import numpy as np
    N = len(correct)
    base_risk = 1.0 - correct.mean()

    print(f"\n  base error rate (coverage=100%): {base_risk:.4f}")
    print(f"\n{'method':>24}  {'AURC':>8}  {'risk@90%':>9}  {'risk@80%':>9}  {'risk@70%':>9}")
    print("─" * 68)

    results = {}
    for name, unc in scores.items():
        order    = np.argsort(unc)            # 확신 높은(불확실 낮은) 것부터
        c_sorted = correct[order]
        risks    = 1.0 - np.cumsum(c_sorted) / np.arange(1, N + 1)
        aurc     = risks.mean()

        def risk_at(cov):
            k = max(1, int(cov * N))
            return 1.0 - c_sorted[:k].mean()

        results[name] = {"aurc": aurc,
                         "risk90": risk_at(0.9),
                         "risk80": risk_at(0.8),
                         "risk70": risk_at(0.7),
                         "risks": risks,
                         "coverages": np.arange(1, N + 1) / N}
        print(f"{name:>24}  {aurc:>8.4f}  {results[name]['risk90']:>9.4f}"
              f"  {results[name]['risk80']:>9.4f}  {results[name]['risk70']:>9.4f}")

    # 판정: 최고 baseline vs 최고 ours
    base_keys = [k for k in scores if "baseline" in k]
    ours_keys = [k for k in scores if "ours" in k]
    best_base = min(results[k]["aurc"] for k in base_keys)
    best_ours_key = min(ours_keys, key=lambda k: results[k]["aurc"])
    best_ours = results[best_ours_key]["aurc"]

    print(f"\n  best baseline AURC = {best_base:.4f}")
    print(f"  best ours AURC     = {best_ours:.4f}  ({best_ours_key})")
    print(f"  → {'OURS WINS' if best_ours < best_base else 'baseline wins'} "
          f"(Δ={best_base - best_ours:+.4f})")
    return results

# ─── Pretty Print ─────────────────────────────────────────────────────────────
def print_field_by_label(d):
    AXES = EpistemicFieldClassifier.AXES
    print(f"{'':>15}" + "".join(f"{a:>14}" for a in AXES))
    print("─" * (15 + 14 * len(AXES)))
    for name, vals in d.items():
        print(f"{name:>15}" + "".join(f"{vals[a]:>14.4f}" for a in AXES))

def print_ood_comparison(r):
    AXES = EpistemicFieldClassifier.AXES
    print(f"\n{'axis':>16}  {'ID mean':>10}  {'OOD mean':>10}  {'OOD - ID':>10}")
    print("─" * 52)
    for ax in AXES:
        id_m, ood_m = r[ax]["id_mean"], r[ax]["ood_mean"]
        flag = "  ← ↑" if ax in ("novelty", "ignorance", "ambiguity") and (ood_m - id_m) > 0.05 else ""
        print(f"{ax:>16}  {id_m:>10.4f}  {ood_m:>10.4f}  {ood_m - id_m:>+10.4f}{flag}")

def print_calibration(r):
    AXES = EpistemicFieldClassifier.AXES
    print(f"\n{'axis':>16}  {'correct':>10}  {'wrong':>10}  {'wrong - correct':>16}")
    print("─" * 58)
    for ax in AXES:
        c, w = r[ax]["correct_mean"], r[ax]["wrong_mean"]
        flag = "  ← ↑" if ax in ("error", "ignorance", "ambiguity", "novelty") and (w - c) > 0.03 else ""
        print(f"{ax:>16}  {c:>10.4f}  {w:>10.4f}  {w - c:>+16.4f}{flag}")


# ─── Main ─────────────────────────────────────────────────────────────────────
from datasets import concatenate_datasets
def main(seed=None, run_failure_typing=True):
    if seed is None:
        seed = CFG["seed"]
    torch.manual_seed(seed)
    import random, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    device = CFG["device"]
    tcfg   = CFG["train"]
    print(f"Device: {device}  |  seed={seed}")

    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    
    print("Loading VitaminC...")
    vc          = load_dataset("tals/vitaminc")
    train_split = vc["train"].select(range(min(tcfg["train_size"], len(vc["train"])))) if tcfg["train_size"] else vc["train"]
    val_split   = vc["validation"].select(range(min(tcfg["val_size"], len(vc["validation"])))) if tcfg["val_size"] else vc["validation"]

    train_ds     = FEVERDataset(train_split, tokenizer, tcfg["max_length"])
    val_ds       = FEVERDataset(val_split,   tokenizer, tcfg["max_length"])
    train_loader = DataLoader(train_ds, batch_size=tcfg["batch_size"], shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

    print("Loading RTE (OOD)...")
    rte        = load_dataset("glue", "rte")
    ood_ds     = OODDataset(rte["validation"], tokenizer, tcfg["max_length"])
    ood_loader = DataLoader(ood_ds, batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)


    model     = EpistemicBERT(**CFG["model"]).to(device)
    model.token_nov.set_special_tokens(tokenizer.all_special_ids)
    model.attn_ign.sep_id = tokenizer.sep_token_id 

    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = build_optimizer(model, tcfg)
    scaler    = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    history = []
    for epoch in range(tcfg["epochs"]):
        print(f"\n═══ Epoch {epoch + 1}/{tcfg['epochs']} ═══")

        tr = train_epoch(model, train_loader, optimizer, scaler, device, tcfg)
        print(f"  train  loss={tr['loss']:.4f}  ce={tr['ce']:.4f}  field={tr['field']:.4f}  acc={tr['acc']:.4f}")
        print(f"  learnable params  margin={F.softplus(model.margin_param).item():.4f}  "
              f"energy_ceil={F.softplus(model.energy_ceiling).item():.4f}  "
              f"con_floor={F.softplus(model.con_energy_floor).item():.4f}  "
              f"truth_temp={F.softplus(model.epistemic.truth_temp).item():.4f}")

        vl = evaluate(model, val_loader, device)
        m  = vl["monitor"]
        print(f"  monitor  field_mean={m['field_mean']}")
        print(f"           field_std ={m['field_std']}")
        print(f"  val    loss={vl['loss']:.4f}  acc={vl['acc']:.4f}")

        print("\n  Epistemic field by label (val):")
        print_field_by_label(vl["field_by_label"])
        print("\n  Support / Counter by label:")
        for name, vals in vl["support_by_label"].items():
            print(f"  {name:>15}  support={vals['support']:.4f}  counter={vals['counter']:.4f}")

        history.append({"epoch": epoch + 1, "train": tr, "val": vl})

    print("\n\n═══ OOD Experiment ═══")
    print_ood_comparison(evaluate_ood(model, ood_loader, val_loader, device))

    print("\n\n═══ Confident-Wrong Analysis ═══")
    evaluate_confident_wrong(model, val_loader, device, top_frac=0.2)

    print("\n\n═══ Identifiability Probe ═══")
    identifiability_probe(model, val_loader, device)

    print("\n\n═══ Ambiguity 진단 ═══")
    diagnose_ambiguity(model, val_loader, device)

    print("\n\n═══ Selective Prediction ═══")
    sp = evaluate_selective_prediction(model, val_loader, device)

    print("\n\n═══ Novelty/Ignorance Disentanglement Probe ═══")
    novelty_ignorance_probe(model, tokenizer, device)

    print("\n\n═══ Attention Entropy Probe (ignorance source 후보) ═══")
    attention_entropy_probe(model, tokenizer, device)

    print("\n\n═══ FEVER Axis ↔ Label Mapping ═══")
    fever_means = evaluate_fever_axes(model, val_loader, device)

    print("\n\n═══ Failure Typing (실패 유형 분리) ═══")
    ft = failure_typing(model, val_loader, tokenizer, device) if run_failure_typing else None

    torch.save(model.state_dict(), "/kaggle/working/epistemic_bert.pt")
    with open("/kaggle/working/results.json", "w") as f:
        json.dump({"history": history}, f, indent=2)

    return model, val_loader, device, ft, fever_means


if __name__ == "__main__":
    import numpy as np
    SEEDS = [7, 123, 42]                      # 3개
    summaries = []
    for s in SEEDS:
        print("\n" + "█" * 70)
        print(f"█  SEED {s}")
        print("█" * 70)
        _, _, _, ft, fever_means = main(seed=s)

        # 유형별 개수 + 핵심 매핑 요약 수집
        if ft is not None:
            types = ft["types"]
            counts = {t: int((types == t).sum())
                      for t in ["ignorance", "ambiguity", "novelty", "metacognitive"]}
        else:
            counts = {}
        # 매핑 top 라벨 (재현 확인용)
        def top_label(axis_i):
            vals = {l: fever_means[l][axis_i] for l in fever_means}
            return max(vals, key=vals.get)
        summaries.append({
            "seed": s,
            "cw_total": int(len(ft["types"])) if ft else 0,
            "type_counts": counts,
            "truth_top": top_label(0),   # 2면 SUPPORTS여야
            "error_top": top_label(1),   # 1이면 REFUTES여야
            "ign_top":   top_label(5),   # 2면 NEI여야
        })

    # ── seed 간 재현 요약 ──
    LAB = {0: "SUP", 1: "REF", 2: "NEI"}
    print("\n\n" + "═" * 70)
    print("재현 요약 (seed 간 일관성)")
    print("═" * 70)
    print(f"{'seed':>6}  {'truth→':>7}  {'error→':>7}  {'ign→':>6}  "
          f"{'cw':>4}  {'ign':>4}  {'amb':>4}  {'nov':>4}  {'meta':>4}")
    print("─" * 60)
    for r in summaries:
        c = r["type_counts"]
        print(f"{r['seed']:>6}  {LAB[r['truth_top']]:>7}  {LAB[r['error_top']]:>7}  "
              f"{LAB[r['ign_top']]:>6}  {r['cw_total']:>4}  "
              f"{c.get('ignorance',0):>4}  {c.get('ambiguity',0):>4}  "
              f"{c.get('novelty',0):>4}  {c.get('metacognitive',0):>4}")

    # 매핑이 3 seed 다 맞았는지
    map_ok = all(r["truth_top"]==0 and r["error_top"]==1 and r["ign_top"]==2 for r in summaries)
    print(f"\n  매핑 재현: {'✓ 3 seed 모두 SUPPORTS→truth, REFUTES→error, NEI→ignorance' if map_ok else '✗ seed마다 다름 — 불안정'}")
    print(f"  유형 분리 재현: ignorance 무리가 3 seed 다 최대 유형인지 표에서 확인")